In [1]:
import os
from pathlib import Path
import json
from collections import Counter

# Ensure paths resolve from repo root regardless of where the kernel starts
_here = Path(os.path.abspath(""))
_root = _here.parent if _here.name == "notebooks" else _here
os.chdir(_root)

from eom.normalise import normalise  # noqa: E402
from eom.schema import EOMDocument  # noqa: E402
from eom.harness import validate  # noqa: E402
from eom.renderers import render_newspaper  # noqa: E402

GOLD = Path("data/gold")

In [2]:
manifest = json.loads((GOLD / "MANIFEST.json").read_text())
print(f"{len(manifest['examples'])} examples")
type_counts = Counter(e["doc_type"] for e in manifest["examples"])
for t, n in type_counts.most_common():
    print(f"  {t:12s} {n}")

32 examples
  memo         5
  other        5
  paper        5
  report       5
  news         4
  policy       4
  transcript   4


In [3]:
passed = 0
total = 0
for ex in manifest["examples"]:
    src = normalise(Path(ex["source"]).read_text(encoding="utf-8"))
    eom = EOMDocument.model_validate_json(Path(ex["eom"]).read_text(encoding="utf-8"))
    report = validate(eom, src)
    if report.passed:
        passed += 1
    total += 1
print(f"Harness pass rate: {passed}/{total} = {passed/total:.0%}")

Harness pass rate: 32/32 = 100%


In [4]:
all_blocks = []
for ex in manifest["examples"]:
    eom = EOMDocument.model_validate_json(Path(ex["eom"]).read_text(encoding="utf-8"))
    all_blocks.extend(eom.blocks)
tier_counts = Counter(b.attention_tier for b in all_blocks)
type_counts = Counter(b.type for b in all_blocks)
print("Tier distribution:", dict(tier_counts))
print("Type distribution:", dict(type_counts))

Tier distribution: {'A': 32, 'B': 69, 'C': 163, 'D': 89}
Type distribution: {'headline': 32, 'lead': 32, 'claim': 37, 'evidence': 83, 'caveat': 38, 'factbox': 57, 'decision': 37, 'appendix': 37}


In [5]:
from IPython.display import HTML, display
seen = set()
for ex in manifest["examples"]:
    if ex["doc_type"] in seen:
        continue
    seen.add(ex["doc_type"])
    eom = EOMDocument.model_validate_json(Path(ex["eom"]).read_text(encoding="utf-8"))
    print(f"\n--- {ex['doc_type']}: {ex['slug']} ---")
    display(HTML(render_newspaper(eom)))


--- memo: 12-factor-app ---



--- news: key-bridge-collapse ---



--- other: curl-readme ---



--- paper: attention-is-all-you-need ---



--- policy: gdpr ---



--- report: doomsday-clock ---



--- transcript: gettysburg-address ---
